In [0]:
%run /Workspace/de_shared/config



In [0]:
BRONZE_PATH = "s3a://medinsight-partd-yash/delta/bronze/partd_raw/"
SILVER_PATH = "s3a://medinsight-partd-yash/delta/silver/prescriber_claims/"
GOLD_PATH   = "s3a://medinsight-partd-yash/delta/gold/territory_scorecard/"

print("Config loaded ✅")

Config loaded ✅


In [0]:
import subprocess
subprocess.run(["pip", "install", "boto3"], capture_output=True)

import boto3
print("boto3 ready ✅")

boto3 ready ✅


In [0]:
s3 = boto3.client(
    's3',
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=AWS_DEFAULT_REGION
)

BUCKET = "medinsight-partd-yash"

# ── 1. Show Delta log history (time travel evidence)
print("=" * 50)
print("DELTA LOG — GOLD TABLE VERSIONS")
print("=" * 50)

response = s3.list_objects_v2(
    Bucket=BUCKET,
    Prefix="delta/gold/territory_scorecard/_delta_log/"
)

for obj in sorted(response.get('Contents', []), key=lambda x: x['Key']):
    size = obj['Size']
    key  = obj['Key'].split('/')[-1]
    print(f"  {key}  ({size} bytes)")

# ── 2. Count files per layer
print("\n" + "=" * 50)
print("FILES PER LAYER ON S3")
print("=" * 50)

for layer, prefix in [
    ("Bronze", "delta/bronze/partd_raw/"),
    ("Silver", "delta/silver/prescriber_claims/"),
    ("Gold",   "delta/gold/territory_scorecard/"),
]:
    resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=prefix)
    count = resp.get('KeyCount', 0)
    size  = sum(o['Size'] for o in resp.get('Contents', []))
    print(f"  {layer:8s}: {count} files, {size/1024/1024:.1f} MB")

# ── 3. Read Gold version 0 (time travel simulation)
print("\n" + "=" * 50)
print("TIME TRAVEL — READING GOLD VERSION 0")
print("=" * 50)

df_gold = (
    spark.read
    .format("delta")
    .option("fs.s3a.access.key", AWS_ACCESS_KEY_ID)
    .option("fs.s3a.secret.key", AWS_SECRET_ACCESS_KEY)
    .option("fs.s3a.endpoint", "s3.amazonaws.com")
    .option("versionAsOf", 0)
    .load("s3a://medinsight-partd-yash/delta/gold/territory_scorecard/")
)

print(f"  Gold version 0 row count: {df_gold.count():,}")
print("  Time travel working ✅")

print("\nPhase 6 complete ✅")


DELTA LOG — GOLD TABLE VERSIONS
  00000000000000000000.crc  (6169 bytes)
  00000000000000000000.json  (5257 bytes)
  00000000000000000001.crc  (6169 bytes)
  00000000000000000001.json  (4710 bytes)
    (0 bytes)

FILES PER LAYER ON S3
  Bronze  : 99 files, 1277.9 MB
  Silver  : 9 files, 1010.6 MB
  Gold    : 9 files, 1056.9 MB

TIME TRAVEL — READING GOLD VERSION 0
  Gold version 0 row count: 23,850,038
  Time travel working ✅

Phase 6 complete ✅
